# Projet ADEME - Gestion de Tournées de Livraison
**TUTEUR : MBM**

**CHEF DE PROJET : Colin**

**EQUIPE : ACHILLE , JENNIFER** 
#  Mots-clés et Définitions

| **Mots clés** | **Définition** |
|:---|:---|
| **ADEME** | Agence de l'Environnement et de la Maîtrise de l'Énergie ; organisme français finançant et promouvant des démonstrateurs et expérimentations pour la transition écologique. |
| **Mobilité multimodale** | Organisation des déplacements combinant plusieurs modes de transport (vélo, bus, train, véhicule utilitaire, etc.) pour optimiser coût, temps et émissions. |
| **Tournée de livraison** | Séquence ordonnée de visites d'un ensemble de points (villes/clients) réalisée par un ou plusieurs véhicules et visant à livrer des objets. |
| **Graphe pondéré** | Modèle mathématique composé de sommets (villes) et d'arêtes (axes routiers) auxquelles sont associés des poids (distances, temps de trajet, coûts). |
| **Problème d’optimisation** | Problème cherchant à minimiser (ou maximiser) une fonction objectif (ex. : distance totale d'une tournée) sous contraintes. |
| **Problème de décision (TSP-DEC)** | Version binaire d'un problème d'optimisation : « Existe-t-il une tournée de coût ≤ K ? ». Utile pour l’étude de complexité. |
| **TSP / VRP** | TSP : *Traveling Salesman Problem* (visiter chaque ville une fois et revenir au départ) ; VRP : *Vehicle Routing Problem* (extension avec flotte, capacités, fenêtres temporelles). |
| **Fenêtre temporelle (time window)** | Intervalle de temps pendant lequel une livraison est autorisée ; livrer en dehors peut être interdit ou pénalisé. |
| **Flotte hétérogène** | Ensemble de véhicules ayant des capacités, coûts ou compatibilités différents (p. ex. différents volumes, types de charge). |
| **Capacités des véhicules** | Contraintes sur le volume/poids que peut transporter un véhicule (souvent modélisées en 2 ou 3 dimensions). |
| **Trafic dynamique** | Variation du temps de parcours d'une arête selon la tranche horaire (matrice de distances dépendant du temps). |
| **Instance (d’un problème)** | Données concrètes définissant un cas à résoudre : nombre de clients, positions, matrice de distances, fenêtres, capacités, etc. |
| **VRPLIB** | Bibliothèque/collection d'instances de référence pour les problèmes de tournées (utilisée pour benchmark et validation scientifique). |
| **Métaheuristique** | Méthode d'optimisation générale (recuit simulé, Recherche Taboue, ALNS, etc.) qui explore l'espace des solutions pour trouver de bonnes solutions approximatives. |
| **PLNE / Modélisation mathématique** | Programmation Linéaire en Nombres Entiers : formulation exacte en variables binaires/entiers pour modéliser contraintes et objectif (utile sur petites instances). |
| **Gap (vs optimal)** | Écart relatif entre le coût obtenu par l'algorithme et le coût optimal de référence, souvent exprimé en pourcentage. |


## Contexte
L’ADEME souhaite optimiser les tournées de livraison pour réduire les émissions de CO₂ et améliorer la logistique urbaine. Le problème consiste à planifier une tournée reliant un ensemble de villes tout en minimisant le temps total de parcours et en respectant diverses contraintes.

## Problématique 
Comment planifier efficacement des tournées de livraison afin de minimiser les distances parcourues et les émissions de CO₂, tout en respectant les contraintes de créneaux horaires et la variation du trafic au cours de la journée ?


# **Modélisation choisie**


Nous avons choisis d'intégrer **deux contraintes**.

1. **Fenêtres temporelles (version avancée)** : Chaque livraison a une fenêtre. La livraison hors-intervalle est interdite, mais l'attente sur place est autorisée (si le véhicule arrive plus tôt).

2. **Trafic dynamique (version simple)**: Les temps de parcours dépendent d'une tranche horaire. Nous modélisons une matrice de temps par tranche (ex : matin/mi-journée/soirée) et utilisons ces valeurs lors de l'évaluation des routes.

**Motivation pédagogique et industrielle** : ces contraintes représentent un cas réaliste (livraison e-commerce en milieu urbain) et restent **implémentables** dans le cadre d'un projet de cycle ingénieur (équilibre entre réalisme et faisabilité).


## 1) Résumé des choix 

- Variante choisie : **CVRPTW avec trafic temporel** (Vehicle Routing Problem with Time Windows + time-dependent travel times).
- Objectif : minimiser **le temps total de tournée** ou alternativement **la date de retour du dernier véhicule** si flotte >1.
- Contraintes : capacité des véhicules , fenêtres temporelles (attente autorisée), et matrice de temps par tranche horaire.
- Format des instances : on partira d'instances VRPLIB et on ajoutera pour chaque client une fenêtre [a_i,b_i] et pour la matrice de temps une discrétisation en S tranches horaires.


## 2) Notation et données 

- Ensembles :
  - $V = \{0,1,\dots,n\}$ avec 0 le dépôt et $1..n$ les clients.
  - $K = \{1,\dots,k\}$ la flotte de véhicules.

- Paramètres :
  - $d^s_{ij}$ ou $t^s_{ij}$ : temps de trajet entre i et j pendant la tranche horaire $s\in S$.
  - $[a_i,b_i]$ : fenêtre temporelle du client i.
  - $q_i$ : demande du client i.
  - $Q_k$ : capacité du véhicule k (on peut supposer $Q_k = Q$ identique pour simplifier).
  - $s_i$ : durée de service au client i.

- Variables de décision :
  - $x^k_{ij} \in \{0,1\}$ : 1 si le véhicule k va directement de i à j.
  - $t_i$ : représente le moment exact d’arrivée au client i, ce qui permet de vérifier le respect des fenêtres temporelles et d’ajuster l’attente éventuelle.
  - $load^k_i$ : charge du véhicule k après visite de i (utile pour démontrer la capacité).


## 3) Formulation mathématique

**Objectif** : minimiser la somme des temps de parcours totaux sur tous les véhicules :

$$\min \sum_{k\in K} \sum_{i\in V} \sum_{j\in V} t^{s(i,j,t_i)}_{ij} \; x^k_{ij}$$

où $t^{s(i,j,t_i)}_{ij}$ désigne le temps de trajet appliqué entre i et j en fonction de la tranche horaire résultant de l'heure $t_i$.
Le coût total est exprimé en minutes de trajet pondérées par tranche horaire, de façon à intégrer la variation du trafic

**Contraintes** :

1. **Conservation des flux (chaque client visité exactement une fois)** :
$$\forall j\in V\setminus\{0\},\quad \sum_{k\in K}\sum_{i\in V} x^k_{ij} = 1$$

2. **Sortie = entrée (pour chaque véhicule et nœud visité)** :
$$\forall k\in K,\forall i\in V,\quad \sum_{j\in V} x^k_{ij} = \sum_{j\in V} x^k_{ji}$$

3. **Capacité** :
$$\forall k,\ \forall r\ \text{route of k},\quad \sum_{i\in r} q_i \le Q_k$$

4. **Fenêtres temporelles (attente autorisée)** : pour tout arc utilisé $x^k_{ij}=1$ :
$$t_j \ge t_i + s_i + t^{s(i,j,t_i)}_{ij} - M(1-x^k_{ij})$$
$$a_j \le t_j \le b_j$$

où $M$ est une constante grande (big-M) pour désactiver la contrainte si l'arc n'est pas utilisé.

5. **Élimination des sous-tours** : mettre une contrainte MTZ (Miller–Tucker–Zemlin) ou des coupes de flot si formulation exacte.

**Remarque** : la dépendance temporelle du coût $t^{s(i,j,t_i)}_{ij}$ rend la formulation exacte complexe (non-linéaire) car les indices de tranche dépendent de $t_i$. Dans la pratique, pour la modélisation et la résolution heuristique, on évaluera le coût d'un arc en fonction de l'heure d'arrivée simulée dans la route (approche trajectoire-simulative).



## 4) Hypothèses pour l'implémentation 

- **Discrétiser** la journée en $S$ tranches (ex : 3 tranches : matin, après-midi, soir). Stocker une matrice $T^s$ pour chaque tranche.
- Lors de l'évaluation d'une route, simuler séquentiellement la tournée et choisir pour chaque arc la valeur $t^{s}_{ij}$ correspondant à la tranche dans laquelle se situe l'heure courante.
- La fonction objectif peut intégrer des pénalités pour violations mineures (si vous choisissez de tolérer).

Ces hypothèses permettront d’assurer la faisabilité des tests et la reproductibilité de nos résultats sur des instances VRPLIB.


## 5) Hypothèses de modélisation et limites d’application

- Les matrices de temps par tranche sont considérées connues et stables pour la période étudiée.
- Les fenêtres temporelles sont strictes : la livraison hors fenêtre est interdite (mais attente autorisée).
- Les demandes $q_i$ sont données et fixes.
- Les véhicules sont initialement identiques (même capacité Q) qu'on pourra enrichir plus tard.
- Les matrices $t^s_{ij}$ sont considérées connues (données historiques ou synthétiques).

**Impact si une hypothèse est violée** :
- Si les matrices de temps sont inexactes → performance réelle peut se dégrader ; prévoir robustesse via scénarios alternatifs.
- Si les fenêtres changent en temps réel → nécessité d'algorithme en ligne / replanification dynamique.

Ces hypothèses définissent donc le cadre de validité du modèle, garantissant un bon équilibre entre réalisme et complexité calculatoire.


## 6) Complexité et justification méthodologique

- Le problème VRP (et VRPTW) est **NP-difficile** : il n’existe pas d’algorithme polynomial connu qui garantisse d’obtenir la solution optimale pour toutes les instances.
- Conséquence : pour des instances réelles (>1000 clients) on privilégiera des **métaheuristiques** (ALNS, Recuit, Taboue) qui trouvent des bonnes solutions en temps raisonnable.
- Pour les petites instances on pourra tester une approche exacte (MIP) pour valider la qualité des heuristiques.

Le problème de tournée de véhicules avec fenêtres temporelles (VRPTW) est une extension du Problème du Voyageur de Commerce (TSP), déjà connu pour être NP-difficile.
En effet, le TSP consiste à trouver le plus court chemin visitant n villes une seule fois, et le VRPTW ajoute plusieurs contraintes supplémentaires : plusieurs véhicules, capacités, fenêtres temporelles et temps variables.
Chaque ajout augmente la complexité combinatoire du problème, ce qui confirme son appartenance à la classe des problèmes NP-difficiles.
Le recuit simulé et la recherche taboue offrent un bon compromis entre qualité de solution et temps de calcul. L’ALNS sera privilégiée pour sa capacité à gérer des contraintes dynamiques.

## 7) Résolution et stratégies d'atténuation des contraintes

### Contrainte 1 : Fenêtres temporelles (attente autorisée)
- **Problème** : fenêtres strictes peuvent rendre beaucoup de solutions invalides et casser certains opérateurs de voisinage.
- **Résolution** :
  1. **Repair operator** : lors d'un voisinage qui produit une solution invalide, appliquer un opérateur de réparation qui réordonne localement les visites pour rétablir les fenêtres.
  2. **Penalisation progressive** : dans une phase d'exploration initiale, autoriser temporairement des violations pénalisées (ajouter un terme de pénalité), puis intensifier la contrainte pour obtenir des solutions faisables.
  3. **Insertion intelligente** : lors de la génération initiale de routes, utiliser heuristiques 'earliest due date' ou 'best insertion respecting windows'.

### Contrainte 2 : Trafic dynamique (matrices par tranche)
- **Problème** : le coût d'un arc dépend de l'heure d'arrivée qui dépend elle-même de la route, rendant l'évaluation coûteuse.
- **Résolution** :
  1. **Simulation rapide** : implémenter une simulation linéaire de la route pour évaluer rapidement le coût temporel d'une route candidate.
  2. **Approximation par moyenne pondérée** : pour des phases rapides, utiliser une estimation moyenne pondérée des temps de trajet (basée sur profil de trafic) pour classer candidatures ; affiner par simulation pour les meilleures solutions.
  3. **Batch re-evaluation** : lorsque l'algorithme explore un voisinage large, réevaluer exactement seulement un sous-ensemble de solutions prometteuses plutôt que toutes.

Ces choix guideront la conception de notre algorithme et serviront de base à la future étude expérimentale.


## 8) Plan de suivi & livrables

- **Sprint 0 (1 semaine)** : finaliser la modélisation (Notebook), décider des formats d'entrée, générer instances tests 5–20 nœuds.
- **Sprint 1 (2 semaines)** : implémentation d'un solveur de base (CVRP simple) + import VRPLIB + tests unitaires.
- **Sprint 2 (2 semaines)** : intégrer fenêtres temporelles (génération d'exemples, heuristique d'insertion), tests et démonstration sur petites instances.
- **Sprint 3 (2 semaines)** : ajouter simulation du trafic dynamique, intégrer au solveur, tests sur instances moyennes (100–500 clients).
- **Sprint 4 (2 semaines)** : étude expérimentale (plan d'expérience, runs=20, courbes, boxplots) et comparaison VRPLIB.
- **Sprint 5 (1 semaine)** : préparation du livrable final + rehearsals soutenance.

**Livrables intermédiaires** : Notebook Modélisation (check), Notebook Implémentation (avec code), Notebook Expérimentation (analyses & plots), slides de soutenance.


## 9) Répartition d'équipe

- Colin (Lead Modélisation) : finalise les formules, rédige la preuve de complexité, maintient ce notebook.
- Jennifer (Data & Instances) : prépare scripts pour adapter VRPLIB, ajoute fenêtres et matrices temporelles.
- Colin (Implémentation) : code du solveur, opérateurs de voisinage, tests unitaires.
- Achille (Expérimentation & Documentation) : plan d'expérience, scripts d'automatisation des runs, plots et rédaction du rapport.

Utilisez un **référentiel Git** partagé et des branches par sprint; chaque livraison/sprint contient des tests de base et un jeu d'instances de référence.


## Plan d'expérimentation (bref, préparatoire)

- Instances de test : utiliser VRPLIB (A-n32-k5, X-n101-k25, M-n200-k17) puis générer des variantes avec fenêtres et matrices temporelles.
- For each instance : 20 runs, measure cost, CPU time, gap vs reference (if available), convergence curves and boxplots.
- Critère cible : gap moyen < 7% pour instances < 200 clients (conforme à l'énoncé).


## Références

- Dantzig & Ramser (1959) — origine du Vehicle Routing Problem.  
- Solomon (1987) — instances et heuristiques pour VRPTW.  
- Ropke & Pisinger (2006) — ALNS, heuristique adaptative pour VRP.
